# Pinecone Vector Database with Gemini Embeddings and Groq LLM

**Environment:** VS Code + local Python / uv

This lesson follows the instructor's Pinecone workflow and preserves all ten sample documents, index setup, insertion, similarity search, metadata filters, score inspection, and retriever creation. It adds detailed explanations and a final RAG extension.

**Provider separation:** Gemini is used only to create embeddings. Groq is used only to generate text answers. Pinecone stores and retrieves the Gemini vectors. The setup and smoke-test cells for Gemini and Groq are separate; only the final RAG chain connects the already-configured retriever and LLM.

**Learning objectives**

- Explain a Pinecone index and why its vector dimension must equal the embedding output dimension.
- Understand `Document.page_content` and `Document.metadata`.
- Generate Gemini embeddings and write them through `PineconeVectorStore`.
- Use similarity search, scores, metadata filters, and retrievers.
- Build a basic grounded RAG answer with Groq.

> API cells make external requests and may consume quotas. Configure your own keys before running them. The notebook does not contain provider secrets.


## 1. Prepare the VS Code environment

Open **Command Prompt (CMD)** in your project directory. If the project already has a `pyproject.toml`, add these dependencies rather than running `uv init` again:

```cmd
uv add langchain-core langchain-pinecone langchain-groq google-genai python-dotenv jupyter ipykernel
uv run jupyter lab
```

Select the kernel belonging to this project in VS Code. `langchain-pinecone` supplies Pinecone integration; avoid installing a separate incompatible legacy `pinecone-client` version manually. If you change packages while Jupyter is running, restart the kernel.


## 2. Import shared components

These imports are provider-neutral LangChain pieces. Keeping them separate from provider setup makes it easier to see which code belongs to document data, the vector-store interface, or the RAG chain.


In [1]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## 3. Load local environment variables

Create a `.env` file in the project root using the accompanying `.env.example` template. `load_dotenv` loads values into `os.environ`; this cell does not print them. Add `.env` to `.gitignore` so secrets are not committed.

This cell only loads configuration. Pinecone, Gemini, and Groq each validate and consume their own key in separate later cells.


In [2]:
loaded_env = load_dotenv(find_dotenv(usecwd=True))
print(".env loaded:", bool(loaded_env))
print("Required environment variable names: PINECONE_API_KEY, GEMINI_API_KEY, GROQ_API_KEY")

.env loaded: True
Required environment variable names: PINECONE_API_KEY, GEMINI_API_KEY, GROQ_API_KEY


### Expected `.env` structure

```dotenv
PINECONE_API_KEY=your_pinecone_key_here
GEMINI_API_KEY=your_gemini_key_here
GROQ_API_KEY=your_groq_key_here
```

Do not copy those placeholder strings as credentials.


## 4. Define the index and embedding configuration

Pinecone's dense-vector field must be created with exactly the same dimension as the vectors returned by the embedding model. We explicitly request **768 dimensions** from Gemini Embedding 2 and create a cosine index with 768 dimensions.

We use a separate index name (`rag-gemini-groq-768`) to avoid accidentally reusing an index previously created with a different embedding model or dimension. If you change the model or dimension later, re-embed all documents and use a matching new index. Embeddings from different models/spaces must not be mixed.


In [3]:
EMBEDDING_MODEL = "gemini-embedding-2"
EMBEDDING_DIMENSION = 768
INDEX_NAME = "rag-gemini-groq-768"
PINECONE_CLOUD = "aws"
PINECONE_REGION = "us-east-1"
PINECONE_METRIC = "cosine"

print({
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": EMBEDDING_DIMENSION,
    "index_name": INDEX_NAME,
    "metric": PINECONE_METRIC,
})

{'embedding_model': 'gemini-embedding-2', 'embedding_dimension': 768, 'index_name': 'rag-gemini-groq-768', 'metric': 'cosine'}


## 5. Pinecone: authenticate and create the client

**Pinecone setup cell — no Gemini or Groq calls here.** The Pinecone SDK client manages the cloud index. The index itself stores vectors and metadata; the `PineconeVectorStore` wrapper that we create later gives it LangChain's standard search methods.


In [4]:
from pinecone import Pinecone, ServerlessSpec
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
if not PINECONE_API_KEY:
    raise RuntimeError(
        "Set PINECONE_API_KEY in your .env file, then rerun this cell.")


pc = Pinecone(api_key=PINECONE_API_KEY)
print("Pinecone client initialized; API key value remains hidden.")

Pinecone client initialized; API key value remains hidden.


## 6. Create or connect to the Pinecone index

An **index** is the named searchable vector resource in Pinecone. The index configuration includes the vector dimension, similarity metric, cloud, and region. For this lesson we use AWS `us-east-1` and cosine similarity.

This cell creates the index only if it does not exist. If it already exists, it checks its dimension and metric before continuing. This prevents accidentally sending 768-dimensional Gemini vectors to an index built for another model's vector size. Do not delete an existing index casually: deletion removes its data.


In [5]:
if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=EMBEDDING_DIMENSION,
        metric=PINECONE_METRIC,
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION),
    )
    print(f"Created Pinecone index: {INDEX_NAME}")
else:
    print(f"Index already exists: {INDEX_NAME}")

index_description = pc.describe_index(INDEX_NAME)
actual_dimension = getattr(index_description, "dimension", None)
actual_metric = getattr(index_description, "metric", None)

if actual_dimension is not None and actual_dimension != EMBEDDING_DIMENSION:
    raise ValueError(
        f"Index dimension is {actual_dimension}, but Gemini is configured for "
        f"{EMBEDDING_DIMENSION}. Choose a new INDEX_NAME or rebuild the index safely."
    )
if actual_metric is not None and actual_metric != PINECONE_METRIC:
    raise ValueError(
        f"Index metric is {actual_metric}, expected {PINECONE_METRIC}. "
        "Choose a compatible index."
    )

index = pc.Index(INDEX_NAME)
print("Index client ready.")

Created Pinecone index: rag-gemini-groq-768
Index client ready.


### Check the index configuration

Inspecting the index helps confirm that the cloud resource matches the intended schema. If this cell fails because the SDK exposes a different description shape in your installed version, inspect `index_description` and compare the server-reported dimension and metric manually; do not ignore a dimension mismatch.


In [6]:
print(index_description)

{'deletion_protection': 'disabled',
 'dimension': 768,
 'host': 'rag-gemini-groq-768-dn2inb7.svc.aped-4627-b74a.pinecone.io',
 'metric': 'cosine',
 'name': 'rag-gemini-groq-768',
 'spec': {'serverless': {'cloud': 'aws', 'region': 'us-east-1'}},
 'status': {'ready': True, 'state': 'Ready'},
 'tags': None,
 'vector_type': 'dense'}


## 7. Gemini: configure the embedding provider

**Gemini-only section.** Embeddings convert texts to dense numerical vectors. Pinecone will index and compare those vectors, but it does not require Groq to create them.

For retrieval using `gemini-embedding-2`, Google's guidance recommends task-aware input formatting. We format documents as `title: none | text: ...` and queries as `task: search result | query: ...`. Each text is embedded in its own request to keep the mapping between one input and one returned vector explicit.


In [7]:
from google.genai import types
from google import genai
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise RuntimeError(
        "Set GEMINI_API_KEY in your .env file, then rerun this cell.")


gemini_client = genai.Client(api_key=GEMINI_API_KEY)
print("Gemini client initialized; API key value remains hidden.")

Gemini client initialized; API key value remains hidden.


### Implement the LangChain `Embeddings` interface

LangChain vector stores expect an embedding object with two methods:

- `embed_documents(texts)`: create vectors for stored passages.
- `embed_query(text)`: create a vector for the user's search query.

The class implements both methods and uses consistent dimensionality. Raising clear errors when the API response is missing vectors prevents bad records being sent to Pinecone.


In [8]:
class GeminiEmbeddings(Embeddings):
    """Gemini Embedding 2 adapter compatible with LangChain vector stores."""

    def __init__(self, client, model_name: str, dimensions: int = 768):
        self.client = client
        self.model_name = model_name
        self.dimensions = dimensions

    @staticmethod
    def _format_document(text: str) -> str:
        return f"title: none | text: {text}"

    @staticmethod
    def _format_query(text: str) -> str:
        return f"task: search result | query: {text}"

    def _embed_one(self, formatted_text: str) -> list[float]:
        response = self.client.models.embed_content(
            model=self.model_name,
            contents=formatted_text,
            config=types.EmbedContentConfig(
                output_dimensionality=self.dimensions
            ),
        )
        if not response.embeddings or not response.embeddings[0].values:
            raise RuntimeError("Gemini returned no embedding values.")
        vector = [float(value) for value in response.embeddings[0].values]
        if len(vector) != self.dimensions:
            raise ValueError(
                f"Expected {self.dimensions} dimensions, received {len(vector)}."
            )
        return vector

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed_one(self._format_document(text)) for text in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed_one(self._format_query(text))


embeddings = GeminiEmbeddings(
    client=gemini_client,
    model_name=EMBEDDING_MODEL,
    dimensions=EMBEDDING_DIMENSION,
)
print("Gemini embedding adapter created.")

Gemini embedding adapter created.


### Smoke-test Gemini embeddings

A smoke test verifies that credentials, model access, and the embedding response are working before we embed all ten documents. It is not a retrieval-quality evaluation. This call may consume API quota.


In [9]:
sample_vector = embeddings.embed_query(
    "What does LangChain help developers build?")
print("Vector dimension:", len(sample_vector))
print("First five values:", sample_vector[:5])

Vector dimension: 768
First five values: [0.014573233, 0.033721987, -0.006313291, 0.029147321, 0.026376808]


## 8. Connect the Gemini embedder to Pinecone through LangChain

The `PineconeVectorStore` adapter connects the Pinecone index to the embedding implementation. When you add documents it calls the embedding model and writes the vectors/text/metadata to Pinecone. When you search with raw text, it embeds the query first and sends the vector to Pinecone.

The explicit `text_key="text"` tells the integration which metadata field stores the original document text. We add the metadata field below when indexing.


In [10]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    text_key="text",
)
print("LangChain PineconeVectorStore ready.")

LangChain PineconeVectorStore ready.


## 9. Prepare the ten instructor documents

A LangChain `Document` has `page_content` plus `metadata`. We preserve the instructor's original ten examples and source labels (`tweet`, `news`, and `website`) because later queries filter on them. Metadata is structured information attached to a record; it is not automatically included in the embedding unless you deliberately include it in the text.


In [11]:
from langchain_core.documents import Document

# The metadata field `source` is intentionally kept consistent across the corpus.
# Pinecone uses it for server-side metadata filtering in later cells.
document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)
document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)
document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)
document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)
document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)
document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)
document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)
document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)
document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)
document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1, document_2, document_3, document_4, document_5,
    document_6, document_7, document_8, document_9, document_10,
]

print(f"Prepared {len(documents)} documents.")
print("First document:", documents[0])

Prepared 10 documents.
First document: page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.' metadata={'source': 'tweet'}


## 10. Insert the documents into Pinecone

`add_documents()` embeds the document text through our Gemini adapter and sends the resulting vectors and metadata to the remote Pinecone index. Stable IDs make rerunning this teaching notebook repeatable: the same IDs upsert the same logical records rather than creating fresh IDs every run.

These IDs are demo identifiers. For a real ingestion pipeline, build IDs from a stable document/chunk key and consider update/delete workflows.


In [12]:
document_ids = [f"demo-document-{i:02d}" for i in range(1, len(documents) + 1)]
upserted_ids = vector_store.add_documents(
    documents=documents, ids=document_ids)
print(f"Submitted {len(upserted_ids)} document IDs to Pinecone.")
print("Example IDs:", upserted_ids[:3])

Submitted 10 document IDs to Pinecone.
Example IDs: ['demo-document-01', 'demo-document-02', 'demo-document-03']


### Verify the index record count

Use Pinecone's index statistics to inspect the vector count. The vector count is a storage check, not a proof that retrieval is relevant. Cloud index updates can take a short time to become visible to queries; if immediate results are missing, verify the index and retry after a short wait.


In [13]:
stats = index.describe_index_stats()
print(stats)
print("Reported vector count:", getattr(
    stats, "total_vector_count", "not exposed by this SDK response"))

{'dimension': 768,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 10}},
 'total_vector_count': 10,
 'vector_type': 'dense'}
Reported vector count: 10


## 11. Direct similarity search with a metadata filter

The instructor's first query uses `similarity_search(query, k=2, filter={"source": "tweet"})`.

- The query text is embedded by Gemini.
- `k=2` requests at most two matching documents.
- `filter` restricts eligible records before returning results; here, only records with `source == "tweet"` are considered.

A filter is different from semantic similarity: it restricts the candidate set using metadata.


In [14]:
query = "LangChain provides abstractions to make working with LLMs easy"
results = vector_store.similarity_search(
    query,
    k=2,
    filter={"source": "tweet"},
)
for rank, doc in enumerate(results, start=1):
    print(f"{rank}. {doc.page_content} [{doc.metadata}]")

1. Building an exciting new project with LangChain - come check it out! [{'source': 'tweet'}]
2. LangGraph is the best framework for building stateful, agentic applications! [{'source': 'tweet'}]


## 12. Similarity search with a score

`similarity_search_with_score()` returns each document together with Pinecone's similarity score. This helps inspect relative ranking, but the score is **not** a probability that the answer is correct. With a cosine index, larger similarity values generally indicate closer vectors. Interpret the score according to this integration and metric rather than assuming every vector store reports scores the same way.


In [15]:
score_results = vector_store.similarity_search_with_score(
    "Will it be hot tomorrow?",
    k=1,
    filter={"source": "news"},
)
for doc, score in score_results:
    print(f"SIMILARITY={score:.4f} | {doc.page_content} | {doc.metadata}")

SIMILARITY=0.7571 | The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees. | {'source': 'news'}


## 13. Inspect the metadata filter by itself

This separate query makes the filter visible and easy to teach. The `source` field is metadata from the `Document`, so Pinecone does not need to infer whether a record is a tweet or a news item from the text.


In [16]:
news_only = vector_store.similarity_search(
    "What happened in the city bank?",
    k=3,
    filter={"source": "news"},
)
print("Returned documents:", len(news_only))
for doc in news_only:
    print(doc.metadata.get("source"), "—", doc.page_content)

Returned documents: 3
news — Robbers broke into the city bank and stole $1 million in cash.
news — The stock market is down 500 points today due to fears of a recession.
news — The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.


## 14. Convert the vector store to a retriever

A **vector store** exposes storage and search operations. A **retriever** is the narrower interface used by chains: given a query, it returns relevant documents.

The instructor uses `search_type="similarity_score_threshold"`, `k=1`, and `score_threshold=0.4`. This is an example threshold, not a universally good value. Tune it against a labelled validation set and be careful to distinguish relevance scores from raw similarity scores.


In [17]:
retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.4},
)
retrieved_docs = retriever.invoke(
    "Stealing from the bank is a crime",
    filter={"source": "news"},
)
for doc in retrieved_docs:
    print(doc.page_content, doc.metadata)

Robbers broke into the city bank and stole $1 million in cash. {'source': 'news'}


## 15. Groq: configure the generation model (separate provider section)

**Groq-only section — no Gemini calls here.** Groq receives natural-language prompts and generates text. It does not create the embeddings or perform Pinecone search; those tasks have already been handled by the Gemini adapter and Pinecone retriever.

The default model below, `openai/gpt-oss-120b`, is an OpenAI open-weight model served through **Groq's API**. The `openai/` part is the model ID, not an OpenAI API call. Change `GROQ_MODEL` if you need a different model currently available to your Groq account.


In [18]:
from langchain_groq import ChatGroq
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise RuntimeError(
        "Set GROQ_API_KEY in your .env file, then rerun this cell.")


GROQ_MODEL = "openai/gpt-oss-120b"
llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
)
print(f"Groq chat model initialized: {GROQ_MODEL}")

Groq chat model initialized: openai/gpt-oss-120b


### Smoke-test Groq without RAG

Test the generation model by itself first. This isolates Groq authentication/model errors from Pinecone retrieval errors. The call may consume your Groq quota.


In [19]:
groq_test = llm.invoke("In one sentence, explain what a vector database does.")
print(groq_test.content)

A vector database stores high‑dimensional vector embeddings and enables fast similarity search to retrieve items whose vectors are nearest to a given query vector.


## 16. Bonus: build an end-to-end RAG chain

This is an extension of the instructor's vector-store lesson. The chain has a retriever fetch context from Pinecone, formats the selected documents, inserts the context and question into a prompt, then asks Groq to generate a grounded answer.

The model setup cells remain separate. This integration cell connects the already-created objects—it does not initialize a new Gemini client or a new Groq model.


### Define how retrieved documents become context

The retriever returns `Document` objects, not one text string. `format_docs()` turns those objects into readable source-labelled context. Keeping source labels helps users inspect evidence and debug retrieval.


In [21]:
from collections.abc import Sequence
from langchain_core.documents import Document


def format_docs(docs: Sequence[Document]) -> str:
    """
    Convert retrieved Document objects into readable,
    source-labelled context for the LLM prompt.
    """
    if not docs:
        return "No relevant documents were retrieved."

    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n"
        f"{doc.page_content}"
        for doc in docs
    )

### Create the answer prompt

`{context}` is populated with retrieved text; `{question}` is the user's question. The instruction tells the model to abstain when the context does not support an answer. This reduces unsupported guesses but does not guarantee the model will never hallucinate.


In [22]:
rag_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer the question using only the provided context. "
        "If the context does not contain the answer, say you do not have enough information.\n\n"
        "Context:\n{context}",
    ),
    ("human", "Question: {question}"),
])

### Compose the LCEL chain

`RunnablePassthrough.assign()` keeps the original question and adds a `context` field computed by the Pinecone retriever. The prompt then feeds the Groq model, and `StrOutputParser` extracts the final text content.


In [23]:
rag_retriever = vector_store.as_retriever(search_kwargs={"k": 3})
rag_chain = (
    RunnablePassthrough.assign(
        context=lambda inputs: format_docs(
            rag_retriever.invoke(inputs["question"])
        )
    )
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG chain ready: question -> Pinecone retrieval -> prompt -> Groq -> text answer")

RAG chain ready: question -> Pinecone retrieval -> prompt -> Groq -> text answer


### Run one question and inspect the evidence

Use a question whose answer is represented in the ten-document sample. The answer comes from Groq; the evidence supplied to it comes from Pinecone. Inspect retrieved documents separately if the answer looks wrong.


In [24]:
question = "What does the sample knowledge base say about LangChain and LangGraph?"
answer = rag_chain.invoke({"question": question})
print("QUESTION:", question)
print("\nANSWER:", answer)
print("\nRETRIEVED SOURCES:")
for doc in rag_retriever.invoke(question):
    print(f"- [{doc.metadata.get('source', 'unknown')}] {doc.page_content}")

QUESTION: What does the sample knowledge base say about LangChain and LangGraph?

ANSWER: The sample knowledge base contains two statements:

- **LangGraph** – It is described as “the best framework for building stateful, agentic applications.”
- **LangChain** – It is mentioned in the context of “building an exciting new project with LangChain – come check it out!”

RETRIEVED SOURCES:
- [tweet] LangGraph is the best framework for building stateful, agentic applications!
- [tweet] Building an exciting new project with LangChain - come check it out!
- [news] The stock market is down 500 points today due to fears of a recession.


## 17. Troubleshooting and interview recap

- **Dimension mismatch:** Pinecone index dimension, embedding output dimensions, and any existing vectors must match. Use a fresh index if changing embedding model/dimension.
- **Authentication:** confirm `PINECONE_API_KEY`, `GEMINI_API_KEY`, and `GROQ_API_KEY` are set in the environment or Colab Secrets. Never print key values.
- **No results after insertion:** inspect index statistics, verify the index name, and allow cloud upserts a short time to become queryable.
- **Unexpected search results:** inspect the query, top-k, metadata filter, source records, and metric; don't tune the LLM prompt before checking retrieval.
- **Scores:** Pinecone similarity scores are not answer correctness probabilities. Thresholds need empirical evaluation.
- **Repeated notebook runs:** stable IDs make this demo's writes repeatable. Avoid deleting a production index just to clear a tutorial.
- **API cost/quotas:** embedding and generation cells call hosted APIs. Run the smallest smoke tests first.

### Interview-ready definitions

- **Pinecone index:** managed searchable resource that stores vectors and metadata for similarity queries.
- **Embedding:** a dense numerical representation of content from an embedding model.
- **Metadata filter:** structured condition restricting which records can match.
- **Top-k:** maximum number of nearest matching records to return.
- **Retriever:** component that accepts a query and returns documents for a downstream chain.
- **RAG:** retrieve relevant evidence first, then condition generation on that evidence.

### Official documentation

- [LangChain Pinecone integration](https://docs.langchain.com/oss/python/integrations/vectorstores/pinecone)
- [Pinecone Python SDK — serverless indexes](https://sdk.pinecone.io/python/how-to/indexes/serverless.html)
- [Gemini Embeddings](https://ai.google.dev/gemini-api/docs/embeddings)
- [LangChain ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq)
